# 01 — Skogsbrand: Kårböle, Ljusdals kommun (Gävleborg)

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/space-data-lab/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_020-Wildfire-Karbole.ipynb)

**Syfte:** Demonstrera hur Sentinel-2-data och AI kan användas för att kartlägga brandskadad skog efter en större skogsbrand. Kårbölebranden 2018 (Ljusdalsbranden) var en av de största skogsbränderna i Sverige i modern tid.

**Partners som bidragit:**
- **Skogsstyrelsen** — referensdata, fältkunskap
- **RISE** — metodutveckling, ImintEngine
- **MSB** (potentiell slutanvändare) — krisberedskap

**Datakällor:**
- Copernicus Sentinel-2 L2A (pre-/post-brand bilder) via Digital Earth Sweden
- Prithvi-EO foundation model (IBM/NASA) — burn scars task head

**Licens:** CC0 1.0 (notebook) · analyzers enligt ImintEngine [LICENSE](https://github.com/TobiasEdman/imintengine)

## 1. Metod

Tre parallella analyser av samma område:

| Analyzer | Output | Vad det visar |
|----------|--------|---------------|
| `spectral` | dNBR (differenced Normalized Burn Ratio) | Brandseveritet per pixel |
| `change_detection` | Multispectral L2-norm diff | Alla förändringar mellan pre/post |
| `prithvi` | Foundation model segmentering | AI-klassificerad brandskada |

**Formel för dNBR:**

$$\text{NBR} = \frac{\text{NIR} - \text{SWIR}}{\text{NIR} + \text{SWIR}} \quad\quad \text{dNBR} = \text{NBR}_{\text{pre}} - \text{NBR}_{\text{post}}$$

Högre dNBR = kraftigare brandskada. USGS klassificerar > 0.66 som *High severity*.

## 2. Setup

In [ ]:
from executors.local import LocalExecutor
import matplotlib.pyplot as plt
import folium

# Kårbölebranden 2018 — Enskogen/Kårböle-området (Ljusdals kommun)
AOI = {
    "west": 15.35,
    "south": 61.78,
    "east": 15.75,
    "north": 62.00,
}
PRE_FIRE = "2018-06-25"   # Före branden
POST_FIRE = "2018-08-30"  # Efter branden

print(f"AOI (Kårböle): {AOI}")
print(f"Pre-fire: {PRE_FIRE}")
print(f"Post-fire: {POST_FIRE}")

## 3. Kör analys

Två `run_job()` — en pre-fire baseline och en post-fire analys. Change detection jämför dem automatiskt.

In [ ]:
executor = LocalExecutor(
    output_dir="outputs/brand_karbole",
    config_path="config/analyzers.yaml",
)

# Post-fire analys (inkluderar change vs. pre-fire baseline)
result = executor.execute(
    date=POST_FIRE,
    coords=AOI,
    baseline_date=PRE_FIRE,
)

print(f"Analyzers: {list(result.analyses.keys())}")
for name, analysis in result.analyses.items():
    print(f"  {name}: {list(analysis.data.keys())}")

## 4. Visualisering

In [ ]:
# Interaktiv Leaflet-karta centrerad på brandområdet
center = [(AOI["south"] + AOI["north"]) / 2, (AOI["west"] + AOI["east"]) / 2]
m = folium.Map(location=center, zoom_start=11, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Satellit",
).add_to(m)
folium.Rectangle(
    bounds=[[AOI["south"], AOI["west"]], [AOI["north"], AOI["east"]]],
    color="#ff0000",
    weight=2,
    fill=False,
    popup="Kårbölebranden 2018 (Ljusdalsbranden)",
).add_to(m)
folium.LayerControl().add_to(m)
m

In [ ]:
# Fyra paneler: RGB pre, RGB post, dNBR, Prithvi seg
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

if hasattr(result, "baseline_rgb") and result.baseline_rgb is not None:
    axes[0, 0].imshow(result.baseline_rgb)
    axes[0, 0].set_title(f"RGB pre-fire ({PRE_FIRE})")
axes[0, 0].axis("off")

axes[0, 1].imshow(result.rgb)
axes[0, 1].set_title(f"RGB post-fire ({POST_FIRE})")
axes[0, 1].axis("off")

if "spectral" in result.analyses and "dnbr" in result.analyses["spectral"].data:
    dnbr = result.analyses["spectral"].data["dnbr"]
    im = axes[1, 0].imshow(dnbr, cmap="RdYlGn_r", vmin=-0.5, vmax=1.0)
    axes[1, 0].set_title("dNBR (högre = allvarligare skada)")
    plt.colorbar(im, ax=axes[1, 0], fraction=0.046)
axes[1, 0].axis("off")

if "prithvi" in result.analyses and "segmentation" in result.analyses["prithvi"].data:
    seg = result.analyses["prithvi"].data["segmentation"]
    axes[1, 1].imshow(seg, cmap="hot")
    axes[1, 1].set_title("Prithvi-EO burn scars")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

## 5. Tolkning

**Vad analysen visar:**
- dNBR-kartan ger en kontinuerlig severitetsskala — värden > 0.66 motsvarar *High severity* enligt USGS
- Prithvi-EO (IBM/NASA foundation model) producerar en binär brandskademask baserad på inlärda spatialtemporala mönster
- Change detection fångar *alla* spektrala förändringar, inte bara brand — användbart för QC

**Jämförelse med referens:**
- Skogsstyrelsens officiella skadeinventering efter Kårbölebranden: ~8 400 ha totalt brandområde
- Denna AOI (~0.4° × 0.22°) täcker huvuddelen men inte hela branden

**Nästa steg:**
- Utöka till hela brandområdet (inkludera Trängslet, Enskogen, Fågelsjö)
- Fler datum: 1, 3, 5 år efter branden → återväxtsanalys via NDVI-trend
- Koppla till MSB:s skadekartor för validering

### Länkar
- [ImintEngine brand-showcase (dashboard)](https://digitalearth.se/case/)
- [ImintEngine/imint/analyzers/spectral.py](https://github.com/TobiasEdman/imintengine/blob/main/imint/analyzers/spectral.py) — dNBR-implementering
- [Prithvi-EO paper](https://arxiv.org/abs/2310.18660)
- [USGS dNBR severity classes](https://burnseverity.cr.usgs.gov/)